# Step 2 - Prototype 1

Here we mainly want to focus on adding the option to have trains going in both directions. In this context, we might have to implement dwell times (to handle buffering) and decide if tracks (all or only some) are directed or undirected. Additionally, we might have to rework the train schedules (more specifically their formulation) to some extent. All in all, this will set us up for longer time horizons.

Also, we might introduce train and station (passenger) weights as more realistic constraints here could potentially solve (or at least mitigate) lazy solver behavior. This could then allow us to get rid of the 'reach station by the end of the time horizon constraint', which might be somewhat problematic in the context of longer time horizons and train schedules with back and forth plans.

To get started, we use the code from the cleaned file for step 1 and iteratively modify it.

In [113]:
# Import Gurobi
import gurobipy as gp
from gurobipy import GRB

import numpy as np

In [114]:
# Initialize the Model
model = gp.Model("Basic_MILP")

### Adjusting the tack blueprint information

Here we also might make some modifications (like maximum allowed speed) but probably also not in this prototype...

In [115]:
# Define the track system using dictionaries to have homogeneous lists and allow for precise track building

# Copy preset for faster building
# {"type": "station", "capacity": 0}
# {"type": "segment", "length": 0, "capacity": 0}

track_blueprint = [
    {"type": "station", "capacity": 3},
    {"type": "segment", "length": 5, "capacity": 1},
    {"type": "station", "capacity": 2},
    {"type": "segment", "length": 2, "capacity": 2},
    {"type": "station", "capacity": 3}
]

In [116]:
# Parsing the track blueprint and generating necessary data (1D lists for Gurobi)
# These store the allowed number of trains on a given block and which blocks are train stations

block_capacities = []
is_station = []

for item in track_blueprint:
    if item["type"] == "station":
        block_capacities.append(item["capacity"])
        is_station.append(True)
    elif item["type"] == "segment":
        # Int conversion not necessary, but gets rid of IDE highlighting
        for _ in range(int(item["length"])):
            block_capacities.append(item["capacity"])
            is_station.append(False)
    else:
        print("typo? No other types yet...")

In [117]:
np.random.randint(0, 3)

2

### Adjusting the train schedules

Here we want to add additional train information (weights related to the number of passengers). Also, but probably not in this prototype, we want to add additional specifications like maximum speed.

**Another idea (possibly for later):** We can add random noise to the stations like

`{"station": 0, "departure": 3, "random_delay": np.random.rand(np.random.randint(0, 3))}`

to our stations and have additional constraints enforcing those to simulate a more dynamic and realistic network. The same could be done for tracks (switches). Generally, in a similar fashion (but maybe without randomness) we could introduce delays or issues into the system and see how the solver adapts the optimal solution then. 

In [118]:
# Similar to our track_blueprint, we use dictionaries for the schedules for a given train
# Especially here, this allows for basically arbitrary additional data that can be used for
# different optimizing goals and constraints. This also nicely handles, that a train doesn't
# have to stop at every station.

# Note: Due to the implementation ('looking-back') departure_times_i has to be >= 1
# Thought: Should we have an (optional) boolean 'is_destination' in (some) station
# descriptions or will this always be clear? Maybe then we could also extract this information
# using a parsing function to create lists / a matrix is_destination...
train_information = [
    # Train 0 (RE-like)
    {
        # Info can be essentially arbitrary and will likely just be used for better optimization goals
        "info": {"passenger_count": 100},
        "schedule": [
            # First journey
            {"station": 0, "departure": 3},
            {"station": 1, "arrival": 10, "departure": 13},
            # (Partially xD) Second journey
            {"station": 2, "arrival": 12, "departure": 20}, # Merged arrival / departure for the turnaround
            {"station": 1, "arrival": 27, "departure": 29},
            {"station": 0, "arrival": 34}
        ]
    },
    # Train 1 (ICE-like)
    {
        "info": {"passenger_count": 250},
        "schedule": [
            # First journey
            {"station": 0, "departure": 5},
            # (Partially xD) Second journey
            {"station": 2, "arrival": 13, "departure": 18, "dwell": 5}, # Merged arrival / departure for the turnaround
            {"station": 0, "arrival": 29}
        ]
    },
    # More trains ...
]

In [119]:
# Number of time steps over which we optimize
time_horizon = 50

# Deriving some additional variables for bounding variables
num_trains = len(train_information)
num_blocks = len(block_capacities)
num_stations = sum(is_station)

# List of all block indices that are stations
station_block_indices = [block for block, station_idx in enumerate(is_station) if station_idx]

# Creating is_near_station List

In [ ]:
# Use is_station list to generate is_near_station

is_near_station = [False] * num_blocks

for j in range(num_blocks):
    
    # Always mark
    if is_station[j]:
        is_near_station[j] = True

        # Check before marking to avoid out of bounds errors
        if j > 0:
            is_near_station[j - 1] = True
        if j < num_blocks - 1:
            is_near_station[j + 1] = True

### Adjusting the decision variable(s)

Instead of only having one 3D-Tensor, we will now have two (one for the forward and one for the backward direction). Using this formulation we will have to adjust most of our constraints. However, this should rather just be considering both tensors instead of actual logical changes.

In [120]:
# Define Decision Variables

# To describe our train movement we need a way to know which train is where at what
# time (-> 3D time-position grid of boolean values)

x_fwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_fwd")
x_bwd = model.addVars(num_trains, num_blocks, time_horizon, vtype=GRB.BINARY, name="x_bwd")

### Constraints (Update this)

#### General Constraints

As general constraints, we have the
- uniqueness constraint (no train can be on multiple tracks at once)
- spawning constraint (a train does not 'exist' until it leaves from the first station in its schedule). For future iterations we should also think about if we want to keep the spawning behavior or replace (or even remove?) it...
- capacity constraints (there can't exist more trains on a block (or station) than its capacity)

#### Movement Constraints

For now, we have the
- 'at most one block per timestep' constraint (will likely be changed in the next phase (step 2) to allow for different train speeds)
- 'departure-time constraint' (trains don't have to stop at stations if they are not on their schedule and also, they can't leave a station until their departure time)

In [121]:
# Testing cell for dictionary access again
i = 0
train_information[i]["schedule"][0]["departure"]

3

In [122]:
# Uniqueness constraint for trains to only exist once on a block (on only a single track at a given time)
# Additionally, we tackle the spawning (departure times), as before a train does not exist on any block
for i in range(num_trains):

    # Extract the spawn time (first departure time)
    spawn_time = train_information[i]["schedule"][0]["departure"]

    for k in range(time_horizon):

        # Sum over all blocks j to ensure that a train only exists once at a given time k
        active_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for j in range(num_blocks))

        # If the train hasn't spawned (or left) yet, it can't exist on the track and we enforce this
        if k < spawn_time:
            model.addConstr(active_tracks == 0, name=f"NotSpawned_Train{i}_Time{k}")
        else:
            # If it already exists, we enforce that the train only exists on one track at any time k
            model.addConstr(active_tracks == 1, name=f"UniquePosition_Train{i}_Time{k}")

In [123]:
# Capacity constraints (the sum over all tracks j and all trains i <= capacities_i)
for k in range(time_horizon):

    # Here we want to sum over all i AND j and add the constraint that this should be smaller than
    # Is this formulation correct and can this be made more compact (yes: x.sum('*', j, k))?
    for j in range(num_blocks):
        occupied_tracks = gp.quicksum(x_fwd[i, j, k] + x_bwd[i, j, k] for i in range(num_trains))

        # Add constraint to enforce capacity maximum across block j
        model.addConstr(occupied_tracks <= block_capacities[j], name=f"Capacity_Block{j}_Time{k}")

In [124]:
# Movement Constraints

# A train can (after spawning) either wait or move
for i in range(num_trains):

    # Extract the spawn station id and spawn time (first departure time)
    spawn_station_id = train_information[i]["schedule"][0]["station"]
    spawn_time = train_information[i]["schedule"][0]["departure"]

    # At every time point, we need to differentiate between different scenarios
    for k in range(time_horizon):

        # If the train hasn't left yet, it's not yet on the grid and the spawn constraint takes care of this
        if k < spawn_time:
            pass
        # Here the train spawns, and we fix a position (the block that corresponds to the first station on
        # the trains schedule)
        elif k == spawn_time:

            # Here we can again use the spawn_station_id and get the one for the next station
            block_index = station_block_indices[spawn_station_id]
            next_station_id = train_information[i]["schedule"][1]["station"]

            # Determine the initial direction of the train
            if spawn_station_id < next_station_id:
                # Moving right -> Spawn in Forward tensor
                model.addConstr(x_fwd[i, block_index, k] == 1, name=f"SpawnFwd_Train{i}_Time{k}_Block{block_index}")
            else:
                # Moving left -> Spawn in Backward tensor
                model.addConstr(x_bwd[i, block_index, k] == 1, name=f"SpawnBwd_Train{i}_Time{k}_Block{block_index}")

        else:

            # Handle out back-looking approach for both channels (tensors)
            for j in range(num_blocks):

                # Forward channel (move 'right')

                # Calculate if this block is a station to allow turning around (switching from Bwd to Fwd)
                turnaround_fwd = x_bwd[i, j, k - 1] if is_station[j] else 0

                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == 0:
                    model.addConstr(x_fwd[i, 0, k] <= x_fwd[i, 0, k - 1] + turnaround_fwd, name=f"MoveFwd_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_fwd[i, j, k] <= x_fwd[i, j, k - 1] + x_fwd[i, j - 1, k - 1] + turnaround_fwd, name=f"MoveFwd_Train{i}_Time{k}_Block{j}")

                # Backward channel (move 'left')

                # Allow switching from Fwd to Bwd
                turnaround_bwd = x_fwd[i, j, k - 1] if is_station[j] else 0

                # If the train is on block one, it must have already been there (j is always ≠ -1)
                if j == num_blocks - 1:
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + turnaround_bwd, name=f"MoveBwd_Train{i}_Time{k}_Block0")
                else:
                    # General back-look constraint (train must have been here or in the previous block)
                    model.addConstr(x_bwd[i, j, k] <= x_bwd[i, j, k - 1] + x_bwd[i, j + 1, k - 1] + turnaround_bwd, name=f"Move_Train{i}_Time{k}_Block{j}")

### Adjusting intermediate (and destination) station constraints and handling

Here we might need / want the information available if a station is a destination station. Also, here we probably need to still do some more modifications to the function and the logic...

In [125]:
# Here we add the constraints for intermediate stations (arrival and departure times and
# no spawning / final destination)

# In this version we only tackle departure times and not dwell times yet (change this?)

# For every train
for i in range(num_trains):

    # Get the trains schedule for easier access
    schedule = train_information[i]["schedule"]

    # Loop through all stops except the spawn station
    for stop_idx in range(1, len(schedule)):
        current_stop = schedule[stop_idx]

        # If there is no departure time, it's the final destination
        if "departure" not in current_stop:
            continue

        # Here we could add another check against malformed schedules (Is this good or
        # might rather cause misunderstanding?)
        if stop_idx + 1 >= len(schedule):
            print(f"Warning: Train {i} has a departure time at its final stop. Double check schedules...")
            continue

        # Extract variables
        departure_time = current_stop["departure"]
        station_id = current_stop["station"]
        station_block = station_block_indices[station_id]

        # Get the next station id to determine the travel direction
        # Here we might encounter issues with malformed schedules
        next_station_id = schedule[stop_idx + 1]["station"]

        # Handle forward direction
        if next_station_id > station_id:
            next_block = station_block + 1

            # Prohibit departure but inhibiting existence on later blocks before t_departure
            if next_block < num_blocks:
                for k in range(departure_time):
                    model.addConstr(x_fwd[i, next_block, k] == 0, name=f"DepWallFwd_Train{i}_Block{next_block}_Time{k}")

        # Handle backward direction
        if next_station_id < station_id:
            next_block = station_block - 1

            # Prohibit departure but inhibiting existence on earlier blocks before t_departure
            if next_block >= 0:
                for k in range(departure_time):
                    model.addConstr(x_bwd[i, next_block, k] == 0, name=f"DepWallBwd_Train{i}_Block{next_block}_Time{k}")

### Adjusting the Objective

Here we have to generally implement changes to use the updated data structure and also use the new information (`"info"`).

In the objective we would like to include the weights and with that, also delays at intermedite stations (also weighted). For this, I would have just for now said:

We assume that the train 'fills' its capacity at the initial station and turnaround stations and then at every station, an equal amount of people get off. This means, that if a train has 100 capacity and goes from 0 -> 1 -> 2 (here turns around) -> 0, the delay factors at the different stations are 0 (we just fill up and depart) -> 1 (50% = 50 get off since we have 2 stations) -> 2 (the other 50% = 50 get off + we 'fill up') -> 0 (all 100% = 100 get off).

We'd likely need some helper variables / lists for this, but they should be easily generatable from the train_information...

In [ ]:
# New Objective (now considering lateness at intermediate stations as well, weighted via passengers getting off)

# For all trains
for i in range(num_trains):

    # Get the schedule and maximum passenger count
    schedule = train_information[i]["schedule"]
    max_passenger_count = train_information[i]["passenger_count"]

    # Loop over the schedule to compute the weights (number of passengers getting off) at each station
    # Might only explicitly give information for 'intermediate' (not spawn or final destination) stations...
    for stop_idx in range(1, len(schedule) - 1):

        # Helper variables ot determine a min / max
        previous_station_id = schedule[stop_idx - 1]
        current_station_id = schedule[stop_idx]
        next_station_id = schedule[stop_idx + 1]

        # Determine if current station is a min or max of these three values. This should give
        # is an is_turnaround indicator (or not if there are no intermediate stations?)
        ...


In [ ]:
# New Objective

# Map the final station ID to the physical block index
destination_keys = [sorted(train_schedules[i].keys())[-1] for i in range(num_trains)]
destination_blocks = [station_block_indices[key] for key in destination_keys]

# Adding an additional constraint to ensure that a train drives to its destination station
# and not allowing the optimizer to cheese its way out of a lateness penalty:
# Force every train to reach its destination by the end of the horizon
for i in range(num_trains):
    dest_block = destination_blocks[i]
    last_t = time_horizon - 1

    model.addConstr(x[i, dest_block, last_t] == 1, name=f"MustFinish_Train{i}")

# Get the target arrival times
target_arrivals = [train_schedules[i][destination_keys[i]]["arrival"] for i in range(num_trains)]

# List to store our terms contributing to the lateness penalty
objective_terms = []

# Loop over all trains
for i in range(num_trains):
    destination_block = destination_blocks[i]
    target_time = target_arrivals[i]

    # We only loop over k starting from the minute after the train is supposed to arrive
    # If they arrive at or before target_time, it is not penalized (adds 0)
    for k in range(target_time + 1, time_horizon):
        # Calculate the positive delay multiplier
        delay_penalty = k - target_time

        # Compute the transition value to check for arrival
        transition = x[i, destination_block, k] - x[i, destination_block, k - 1]

        # Multiply by the arrival transition at the destination block and add to our objective list
        objective_terms.append(delay_penalty * transition)

model.setObjective(gp.quicksum(objective_terms), GRB.MINIMIZE)